<h1> SHAP-Analysis – Auto-PyTorch</h1>

Before starting the script, you need to initialise the persistant_predict_server.py file.
Check your Root folder & your specific Auto-PyTorch DockerID

docker cp ROOTFOLDER\persistent_predict_server.py DOCKERID:/workspace/persistent_predict_server.py
docker start DOCKERID

In [1]:
import os
import json
import subprocess
import uuid
import time

import numpy as np
import shap

c:\Users\PE-GM\Anaconda\envs\visana\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
FRAMEWORK = "auto-pytorch"

HOST_BASE = r"E:\StudiumMasterarbeit"
SAVED_MODELS_DIR = os.path.join(HOST_BASE, "saved_models", "auto-pytorch")

CONTAINER_ID = "b5220e80aee9"
PERSISTENT_SERVER_IN_CONTAINER = "/workspace/persistent_predict_server.py"
CONTAINER_MOUNT_PREFIX = "/workspace/oberer-ordner"
TMP_DIRNAME = "tmp_shap_bridge"

N_BACKGROUND = 50
N_EXPLAIN = 30
SKIP_IF_RESULTS_EXIST = True

OUTPUT_DIR = os.path.join(HOST_BASE, "shap_results", FRAMEWORK)
TMP_DIR = os.path.join(HOST_BASE, TMP_DIRNAME)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

tasks = sorted(
    d for d in os.listdir(SAVED_MODELS_DIR)
    if os.path.isdir(os.path.join(SAVED_MODELS_DIR, d))
)

def container_path(host_path):
    return CONTAINER_MOUNT_PREFIX + "/" + os.path.relpath(
        host_path, HOST_BASE
    ).replace("\\", "/")

print(f"{len(tasks)} Aufgaben gefunden.")


In [ ]:
# SHAP-Werte berechnen als Loop, der durch alle Aufgaben iteriert. Ergebnisse werden in OUTPUT_DIR gespeichert.
import cmd


for task_name in tasks:
    task_dir = os.path.join(SAVED_MODELS_DIR, task_name)
    out_dir = os.path.join(OUTPUT_DIR, task_name)

    # Bereits berechnete Aufgaben überspringen
    if SKIP_IF_RESULTS_EXIST and os.path.exists(
        os.path.join(out_dir, "shap_values.npy")
    ):
        print(f"[{task_name}] übersprungen – Ergebnis existiert bereits.")
        continue

    os.makedirs(out_dir, exist_ok=True)

    print(f"\n[{task_name}]")

    with open(os.path.join(task_dir, "feature_names.json")) as f:
        feature_names = json.load(f)

    X_train = np.load(os.path.join(task_dir, "X_train.npy"))
    X_test = np.load(os.path.join(task_dir, "X_test.npy"))

    model_container = container_path(os.path.join(task_dir, "model.joblib"))

    background = shap.sample(
        X_train, min(N_BACKGROUND, len(X_train)), random_state=42
    )
    X_explain = X_test[:min(N_EXPLAIN, len(X_test))]
    max_evals = max(500, 2 * len(feature_names) + 1)

    proc = subprocess.Popen(
        [
            "docker", "exec", "-i", CONTAINER_ID,
            "python3", "-u",
            PERSISTENT_SERVER_IN_CONTAINER,
            model_container
        ],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE, #stderr is redirected to stdout in the persistent server, so we can capture it here
        text=True,
        bufsize=1
    )

    try:
        if proc.stdout.readline().strip() != "READY":
            raise RuntimeError("Predict-Server nicht bereit.")

        def predict(X):
            uid = uuid.uuid4().hex
            host_in = os.path.join(TMP_DIR, f"in_{uid}.npy")
            host_out = os.path.join(TMP_DIR, f"out_{uid}.npy")
            host_done = host_out + ".done"
            host_err = host_out + ".err"
            try:
                np.save(host_in, np.asarray(X))
                proc.stdin.write(f"{container_path(host_in)}\t{container_path(host_out)}\n")
                proc.stdin.flush()

                timeout = 300
                start = time.time()
                while True:
                    if os.path.exists(host_err):
                        with open(host_err) as f:
                            msg = f.read()
                        raise RuntimeError(f"Predict-Server error: {msg}")
                    if os.path.exists(host_done):
                        break
                    if proc.poll() is not None:
                        err = proc.stderr.read() if proc.stderr else ""
                        raise RuntimeError(f"Predict-Server died.\nstderr: {err[-2000:]}")
                    if time.time() - start > timeout:
                        raise TimeoutError(f"Predict-Server timed out after {timeout}s on this batch")
                    time.sleep(0.2)

                return np.load(host_out)
            finally:
                for p in (host_in, host_out, host_done, host_err):
                    if os.path.exists(p):
                        os.remove(p)


        print(
            f"  {len(X_train)} Trainingsdaten | "
            f"{len(X_explain)} zu erklärende Samples | "
            f"{len(feature_names)} Merkmale"
        )

        explainer = shap.Explainer(
            predict,
            background,
            feature_names=feature_names,
            seed=42
        )
        shap_values = explainer(X_explain, max_evals=max_evals)

        # Efficiency-Check: SHAP-Werte + Base Value sollten der
        # Modellvorhersage entsprechen.
        y_pred_explained = predict(X_explain)
        base_values = np.asarray(shap_values.base_values)
        efficiency_gap = y_pred_explained - (
            shap_values.values.sum(axis=1) + base_values
        )

        print(
            f"  Efficiency-Check: mean|gap| = "
            f"{np.mean(np.abs(efficiency_gap)):.6f}"
        )

    finally:
        try:
            proc.stdin.write("EXIT\n")
            proc.stdin.flush()
        except Exception:
            pass

        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.terminate()
            proc.wait(timeout=10)

    np.save(os.path.join(out_dir, "shap_values.npy"), shap_values.values)
    np.save(os.path.join(out_dir, "X_explained.npy"), X_explain)
    np.save(os.path.join(out_dir, "base_values.npy"), base_values)
    np.save(os.path.join(out_dir, "y_pred_explained.npy"), y_pred_explained)

    with open(os.path.join(out_dir, "feature_names.json"), "w") as f:
        json.dump(list(feature_names), f)

    print(f"  Gespeichert: {out_dir}")
